# Laboratorio 5

### Simulaciones Basadas en Modelos Bid y Choice de Localización

Ayudante: Janus Leonhardt | jaleonhardt@uc.cl 

Profesor: Ricardo Hurtubia | rhurtubia@uc.cl

## Objetivos del Laboratorio

1. Generar una población sintética de agentes habitacionales con categorías para su uso en los enfoques *bid* y *choice*.

2. Para el modelo Choice, extraer los coeficientes estimados con `xlogit`, calcular la utilidad para cada observación en cada celda y sus alternativas, calcular las probabilidades, plotearlas y realizar una simulación de Monte Carlo para localizar a un solo agente.

3. Para el modelo Bid, extraer los coeficientes estimados con `biogeme`, calcular la disposición a pagar por celda entre las tres categorías de hogares, calcular las probabilidades, plotearlas y realizar una simulación de Monte Carlo para ejemplificar cómo se puja cada celda.

---

## 1. Repaso de Laboratorio 4

En el **Laboratorio 4**, aprendimos a estimar modelos de localización basados en un enfoque *bid* y *choice* con las librerías `xlogit` y `biogeme` de Python. En este laboratorio, usaremos los resultados de los modelos estimados previamente para simular la localización de nuevos agentes en nuestra área de estudio, asumiendo que tenemos información de las celdas al año base de simulación.

---

## 2. Instalación y Configuración del Entorno de Trabajo

### Requisitos Previos

- Python 3.7 o superior

- Jupyter Notebook

- **Librerías de Python**:

  - pandas  
  - geopandas  
  - shapely  
  - h3  
  - matplotlib  
  - folium  
  - mapclassify  
  - seaborn
  - requests
  - xlogit
  - biogeme

#### Instalación de Librerías

Recordemos que las librerías deben haber quedado correctamente instaladas en nuestro entorno virtual (`.conda` o `.venv`) creado en el Laboratorio 1. En caso contrario, se debe ejecutar la siguiente línea de código.

`!pip install pandas geopandas shapely h3 matplotlib folium mapclassify seaborn requests xlogit biogeme`

---

## 3. Carga de Datos Previos

En este laboratorio, partiremos desde los archivos generados en los Laboratorio 2, 3 y 4, pero adaptados a la estructura que necesitamos para implementar correctamente los modelos de localización respectivos. Así, los archivos que necesitaremos serán los siguientes.

1. **`celdas_simulacion.geojson`** (basado en el **Laboratorio 3**): Contiene información agregada por celda *h3* para el año de inicio del período de simulación (por ejemplo, 2022), incluyendo las características del entorno construido, las accesibilidades gravitacionales del Laboratorio 3 y los datos externos del Laboratorio 2. 

2. **`choice.pickle`**: Contiene los resultados del modelo Choice estimado en el Laboratorio 4, en particular las variables explicativas y los betas estimados.

2. **`bid.pickle`**: Contiene los resultados del modelo Bid estimado en el Laboratorio 4, en particular las variables explicativas y los betas estimados.

La idea es que estos datos sean **coherentes** con el **año de inicio** del período de simulación. Para el ejemplo, veremos cuáles construcciones nuevas aparecieron desde 2022, y estudiaremos cómo eligen su localización con base en las características de las celdas en 2022 (contenidas en `celdas_simulacion.geojson`).

In [ ]:
import geopandas as gpd

# cargar celdas de simulación
celdas_simulacion = gpd.read_file('celdas_simulacion.geojson')

In [ ]:
import pickle
import pandas as pd

# cargar el modelo choice desde el archivo .pickle
with open("../lab_04/choice.pickle", "rb") as file:
    choice_model = pickle.load(file)

# cargar el modelo bid desde el archivo .pickle
with open("../lab_04/bid.pickle", "rb") as file:
    bid_model = pickle.load(file)

# crear un diccionario con los coeficientes del modelo choice
betas_choice = {str(name): float(beta) for name, beta in zip(choice_model.coeff_names, choice_model.coeff_)}

# crear un diccionario con los coeficientes del modelo bid
betas_bid = {str(key[2:]): float(value) for key, value in bid_model.get_beta_values().items()}

# ver los coeficientes del modelo choice y bid
print(betas_choice)
print(betas_bid)

---

## 4. Generación de Población Sintética

En este laboratorio, generaremos una población sintética de 1.000 agentes habitacionales clasificados en tres categorías de hogares. Esto deberá ajustarse a los requerimientos específicos de cada caso de estudio, con respecto al total de agentes a simular y sus caracterísiticas respectivas.

In [ ]:
import numpy as np

# definir el número de agentes sintéticos
num_agentes = 1000

# definir las categorías de viviendas
categorias_viviendas = [1, 2, 3]

# definir proporciones para cada categoría
proporciones = [0.5, 0.3, 0.2] 

# generar las categorías para cada agente
np.random.seed(42)
categorias = np.random.choice(categorias_viviendas, size=num_agentes, p=proporciones)

# crear un DataFrame para la población sintética
poblacion_sintetica = pd.DataFrame({'id_agente': range(1, num_agentes + 1),'categoria_vivienda': categorias})

# ver las primeras filas de la población sintética
poblacion_sintetica.head()

---

## 5. Simulación con el Modelo Choice

En esta sección, utilizaremos el modelo Choice para calcular las utilidades y probabilidades de elección de localización para cada agente en cada celda. Posteriormente, y a modo de ejemplo, realizaremos una simulación de Monte Carlo para asignar una localización a un agente específico.

### Preparación de la Base de Datos Choice para Simular

A continuación, se generan todas las combinaciones posibles entre la población sintética y las celdas de simulación para calcular las utilidades. Esto debe ajustarse a la disponibilidad efectiva de celdas en el área de estudio respectiva.

In [ ]:
# añadir una clave temporal para realizar el producto cartesiano
poblacion_sintetica['key'] = 1
celdas_simulacion['key'] = 1

# crear el conjunto de elección (producto cartesiano entre agentes y celdas)
bd_choice = pd.merge(poblacion_sintetica, celdas_simulacion, on='key').drop('key', axis=1)

# definir las variables explicativas utilizadas en el modelo Choice
variables_explicativas_choice = [
    'acc_auto_m2_comercio', 
    'acc_auto_m2_departamento',
    'acc_auto_n_deporte_y_recreacion', 
    'acc_auto_n_educacion_y_cultura',
    'acc_auto_m2_habitacional', 
    'acc_auto_m2_industria',
    'acc_auto_m2_oficina', 
    'acc_auto_n_salud']

# definir una función para calcular la utilidad
def calcular_utilidad_choice(row, betas):
    utilidad = 0
    for var in variables_explicativas_choice:
        utilidad += betas.get(var, 0) * row[var]
    return utilidad

# calcular la utilidad para cada fila
bd_choice['utilidad'] = bd_choice.apply(lambda row: calcular_utilidad_choice(row, betas_choice), axis=1)

# calcular la probabilidad de elección para cada fila
bd_choice_final = []
for index, agente in poblacion_sintetica.iterrows():
    bd_choice_ = bd_choice[bd_choice['id_agente'] == agente['id_agente']]
    bd_choice_['probabilidad'] = np.exp(bd_choice_['utilidad']) / np.exp(bd_choice_['utilidad']).sum()
    bd_choice_final.append(bd_choice_)

# concatenar los resultados
bd_choice = pd.concat(bd_choice_final)

### Ejemplo de Probabilidades de Localización

In [ ]:
# filtrar las filas de la base de datos de elección para el agente 1
bd_choice_agente_1 = bd_choice[bd_choice['id_agente'] == 1]

# convertir a GeoDataFrame
bd_choice_agente_1 = gpd.GeoDataFrame(bd_choice_agente_1, geometry='geometry')

# plotear las probabilidades de elección para el agente 1
bd_choice_agente_1.explore(column='probabilidad', scheme='NaturalBreaks', k=10)

### Ejemplo de Simulación de Localización (por Monte Carlo)

In [ ]:
# definir la semilla y simular una elección para el agente 1
np.random. seed(18)
choice_agente_1 = np.random.choice(bd_choice_agente_1['h3'], p=bd_choice_agente_1['probabilidad'])

# ver la celda seleccionada
celdas_simulacion[celdas_simulacion['h3']==choice_agente_1].explore()

Con base en lo anterior, es posible simular la localización de los $n$ agentes del **DataFrame de Población Sintética**. Es importante recordar que **las variables explicativas (como las accesibilidades a distintos usos de suelo) deben actualizarse periódicamente durante la simulación**. Esto se debe a que las nuevas localizaciones de los agentes modificarán dichas variables, lo que, a su vez, impactará los atributos del entorno construido y la atractividad de cada celda.

---

## 6. Simulación con el Modelo Bid

En esta sección, utilizaremos el modelo Bid para calcular la disposición a pagar ($WTP$) de cada categoría de hogar por cada celda. Posteriormente, calcularemos las probabilidades de que cada categoría gane la subasta por una celda específica y realizaremos una simulación de Monte Carlo para asignar una categoría a una celda.

### Preparación de la Base de Datos Bid para Simular

A continuación, se generan todas las combinaciones posibles entre la población sintética y las celdas de simulación para calcular las disposiciones a pagar por cada categoría. Esto debe ajustarse a la disponibilidad efectiva de las celdas en el área de estudio respectiva.

In [ ]:
# definir las variables explicativas utilizadas en el modelo Bid
variables_explicativas_bid = [
    'acc_auto_m2_comercio', 
    'acc_auto_n_educacion_y_cultura',
    'acc_auto_m2_habitacional', 
    'acc_auto_m2_industria',
    'acc_auto_n_salud']

# crear un DataFrame que une cada celda con cada categoría de vivienda
categorias = [1, 2, 3]
bd_bid = celdas_simulacion.copy()

# crear una columna para cada categoría y sus variables
for c in categorias:
    for var in variables_explicativas_bid:
        bd_bid[f'{var}_H{c}'] = bd_bid[var]

# ver las primeras filas de la base de datos
bd_bid.head()

In [ ]:
# calcular la disposición a pagar (WTP) para cada categoría y cada celda
def calcular_WTP_bid(row, categoria, betas):
    WTP = 0
    for var in variables_explicativas_bid:
        coef = betas.get(f'{var}_H{categoria}', 0)
        WTP += coef * row[var]
    return WTP

# aplicar la función para cada categoría
for c in categorias:
    bd_bid[f'WTP_H{c}'] = bd_bid.apply(lambda row: calcular_WTP_bid(row, c, betas_bid), axis=1)

# sumar las disposiciones a pagar
bd_bid['WTP_sum'] = bd_bid['WTP_H1']+bd_bid['WTP_H2']+bd_bid['WTP_H3']

# ver una muestra de las disposiciones a pagar
bd_bid[['h3', 'WTP_H1', 'WTP_H2', 'WTP_H3', 'WTP_sum']].head()

In [ ]:
# calcular las probabilidades de cada categoría por celda
bd_bid['prob_H1'] = bd_bid.apply(lambda row: np.exp(row['WTP_H1']) / (np.exp(row['WTP_H1']) + np.exp(row['WTP_H2']) + np.exp(row['WTP_H3'])), axis=1)
bd_bid['prob_H2'] = bd_bid.apply(lambda row: np.exp(row['WTP_H2']) / (np.exp(row['WTP_H1']) + np.exp(row['WTP_H2']) + np.exp(row['WTP_H3'])), axis=1)
bd_bid['prob_H3'] = bd_bid.apply(lambda row: np.exp(row['WTP_H3']) / (np.exp(row['WTP_H1']) + np.exp(row['WTP_H2']) + np.exp(row['WTP_H3'])), axis=1)

# ver una muestra de las probabilidades
bd_bid[['h3', 'prob_H1', 'prob_H2', 'prob_H3']].head()

### Ejemplo de Probabilidades de Elección de Categorías

In [ ]:
# plotear las probabilidades de elección de categoría 1 (prob_H1: 'alta calidad')
bd_bid.explore(column='prob_H1', scheme='NaturalBreaks', k=10)

In [ ]:
# plotear las probabilidades de elección de categoría 2 (prob_H2: 'media calidad')
bd_bid.explore(column='prob_H2', scheme='NaturalBreaks', k=10)

In [ ]:
# plotear las probabilidades de elección de categoría 3 (prob_H3: 'baja calidad')
bd_bid.explore(column='prob_H3', scheme='NaturalBreaks', k=10)

### Ejemplo de Simulación de "Puja" (por Monte Carlo)

In [ ]:
# filtrar las filas de la base de datos de elección para la celda 1947
bd_bid_celda_1 = bd_bid.iloc[1947:1948]

# simular una elección de categoría para la celda 1947
np.random.seed(60)
bid_celda_1 = np.random.choice(categorias, p=bd_bid_celda_1[['prob_H1', 'prob_H2', 'prob_H3']].values[0])

# ver la categoría seleccionada
print(f'H{bid_celda_1}')

# plotear la celda 1
bd_bid_celda_1.explore()

Con esto completamos el Laboratorio 5 sobre simulación de localizaciones basadas en los enfoques *bid* y *choice*.

**Fin del Laboratorio 5**

---

### 7. Anexo – Ejemplo Simple de Motor de Simulación

En este anexo construiremos un ejemplo muy simple de motor de simulación basado en el modelo *Choice* del Laboratorio 5. La idea es mostrar cómo: (i) asignar capacidad por celda y por categoría de vivienda; (ii) localizar agentes por Monte Carlo, usando las probabilidades calculadas; (iii) actualizar los atributos de las celdas cada vez que se localiza un agente; y (iv) recalcular accesibilidades después de localizar a un cierto número de agentes.

**Importante:** En este ejemplo, el “copamiento” de las celdas se hace solo por número de agentes. En la práctica, tiene mucho más sentido hacerlo por m² construidos (o por alguna medida de densidad) definida a partir del Plan Regulador Comunal o de una regla acordada por el grupo de trabajo.

**Paso 1 — Capacidad por Celda**

Partimos creando una copia de `celdas_simulacion` para no modificar el original. Luego, definimos una capacidad máxima de agentes por celda y por categoría de vivienda. En este ejemplo, usaremos las categorías de vivienda: 1, 2 y 3 (pese a que en el ejemplo tenemos un único modelo para las tres categorías) y asignaremos la misma capacidad a todas las celdas (por ejemplo, 10 agentes por celda y categoría). En un caso real, esta capacidad podría calcularse en función de m² útiles, densidades permitidas por el Plan Regulador u otros criterios normativos.

In [ ]:
# hacemos una copia para el ejemplo de simulación
celdas_sim = celdas_simulacion.copy()

# categorías de vivienda (las mismas que usamos en la población sintética)
categorias_viviendas = [1, 2, 3]

# capacidad de ejemplo: mismo número máximo de agentes por celda y categoría
capacidad_ejemplo = 5  # ejemplo, en la práctica esto debiera ser por m2 o norma del Plan Regulador

# crear columnas de capacidad y ocupación inicial por categoría
for cat in categorias_viviendas:
    celdas_sim[f'cap_H{cat}'] = capacidad_ejemplo
    celdas_sim[f'ocupados_H{cat}'] = 0  # partimos sin agentes localizados, pero esto puede reflejar la ocupación actual

# tamaño promedio de vivienda por categoría (ejemplo)
m2_por_categoria = {
    1: 40,   # hogares H1
    2: 80,   # hogares H2
    3: 150   # hogares H3
}

**Paso 2 - Motor Simple de Simulación**

In [ ]:
# función auxiliar para normalizar probabilidades de forma segura
def normalizar_probabilidades(p):
    p = np.array(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0)
    s = p.sum()
    if s > 0:
        return p / s
    # si la suma es 0, repartimos uniforme
    return np.ones_like(p) / len(p)

# número de agentes a simular
n_agentes_simular = 200

# tomamos los primeros n agentes (o podrías usar .sample())
agentes_a_simular = poblacion_sintetica.iloc[:n_agentes_simular].copy()

# lista para guardar resultados de localización
resultados_simulacion = []

for index, agente in agentes_a_simular.iterrows():
    id_agente = agente['id_agente']
    cat_viv = agente['categoria_vivienda']
    
    # alternativas de este agente desde bd_choice
    bd_agente = bd_choice[bd_choice['id_agente'] == id_agente].copy()
    
    # unimos con capacidad y ocupación actuales para su categoría
    bd_agente = bd_agente.merge(
        celdas_sim[['h3', f'cap_H{cat_viv}', f'ocupados_H{cat_viv}']],
        on='h3',
        how='left'
    )
    
    # filtramos solo celdas que aún tienen cupo para esta categoría
    bd_agente = bd_agente[bd_agente[f'ocupados_H{cat_viv}'] < bd_agente[f'cap_H{cat_viv}']]
    
    # si no quedan celdas disponibles, el agente queda sin localizar en esta iteración
    if bd_agente.empty:
        resultados_simulacion.append({'id_agente': id_agente, 'categoria_vivienda': cat_viv, 'h3_elegida': None})
        continue
    
    # renormalizar probabilidades sobre las celdas disponibles
    p = normalizar_probabilidades(bd_agente['probabilidad'].values)
    h3_opciones = bd_agente['h3'].values
    
    # elegir una celda por Monte Carlo
    h3_elegida = np.random.choice(h3_opciones, p=p)
    
    resultados_simulacion.append({
        'id_agente': id_agente,
        'categoria_vivienda': cat_viv,
        'h3_elegida': h3_elegida
    })
    
    # actualizar ocupación y atributos de la celda elegida
    mascara = celdas_sim['h3'] == h3_elegida
    
    # aumentar el número de ocupados de esa categoría
    celdas_sim.loc[mascara, f'ocupados_H{cat_viv}'] += 1
    
    # aumentar el número de viviendas (ejemplo: n_habitacional crece en 1 por agente)
    if 'n_habitacional' in celdas_sim.columns:
        celdas_sim.loc[mascara, 'n_habitacional'] += 1
    
    # aumentar m2 habitacionales según la categoría (ejemplo simple)
    if 'm2_habitacional' in celdas_sim.columns:
        celdas_sim.loc[mascara, 'm2_habitacional'] += m2_por_categoria.get(cat_viv, 0)

In [ ]:
celdas_sim.explore(column='ocupados_H3', scheme='NaturalBreaks', k=10, title='Ocupación H1 tras simulación')

**Paso 3 - Función para Recalcular Accesibilidades**

In [ ]:
# función para calcular accesibilidades
def calcular_accesibilidades(celdas, variables, impedance_matrix, prefix):
    accesibilidades = impedance_matrix.dot(celdas[variables])
    accesibilidades.columns = [f"{prefix}_{col}" for col in accesibilidades.columns]
    accesibilidades = celdas[['h3', 'geometry']].merge(
        accesibilidades,
        left_index=True,
        right_index=True,
        how='left'
    )
    return accesibilidades

**Paso 4' - Recalcular Probabilidades**

En este último paso, luego de localizar a un cierto número de agentes (por ejemplo, cada 50 o 100 iteraciones), se deben actualizar las accesibilidades y volver a calcular las probabilidades de elección. Para ello, primero se trabajan las columnas de celdas_sim que participan en el modelo (por ejemplo, m2_habitacional, n_habitacional u otras variables de interés) y se recalculan las accesibilidades gravitacionales con la función calcular_accesibilidades, utilizando la misma matriz de impedancias del Laboratorio 3. Una vez actualizadas las columnas de accesibilidad en celdas_sim, se genera una nueva base de elección bd_choice (o se actualiza la existente), se recalculan las utilidades con los betas del modelo Choice y se obtienen nuevamente las probabilidades de localización para cada agente y celda. De este modo, cada “bloque” de la simulación incorpora el efecto acumulado de las localizaciones previas sobre el entorno construido y sobre la atractividad relativa de las celdas.